# Redshift DataSet Regressor: Linear Regression

In this notebook we will build a linear regression model using the redshift dataset.  Let's dive in! 

## import libraries

In [ ]:
# arrays
import numpy as np

# unpacking files
import tarfile

# fits
from astropy.io import fits
from astropy.utils.data import download_file
from astropy.table import Table
import pandas as pd

# plotting
from matplotlib import pyplot as plt

# sklearn 
from sklearn.model_selection import train_test_split #, RandomizedSearchCV, validation_curve
#from sklearn.model_selection import KFold, cross_validate
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline 

## Read and Inspect Data

First we will import our data and convert it into a pandas dataaframe

In [ ]:
file_url = 'https://archive.stsci.edu/missions/hlsp/3d-hst/RELEASE_V4.0/Photometry/3dhst_master.phot.v4.1.tar'
tarfile.open(download_file(file_url, cache=True), "r:").extract('3dhst_master.phot.v4.1/3dhst_master.phot.v4.1.cat', '.')
tab = Table.read('3dhst_master.phot.v4.1/3dhst_master.phot.v4.1.cat', format='ascii').to_pandas()
tab.head()

Next, we can start to inspect our data.

In [ ]:
tab.info()

## Clean up 

Before building and applying a regression model, we first need to inspect and clean the dataset. This process will explore the data we have, and filter out thing like missing data or data not relevant to our task.

To explore the physical parameters of the sample, we plot the spectroscopic redshift vs. the mass derived from the FAST phototmetric fit:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

ax.scatter(tab.z_spec, tab.lmass, alpha=0.2, color='grey')
ax.set_xlim(0, 2)
ax.set_ylim(7, 12)
ax.set_xlabel(r'$z_{\rm spec}$')
ax.set_ylabel(r'$\log{(M_{*})}\,\,[M_{\odot}]$');

### Use domain specific knowledge to filter data

Below we filter out low mass galaxies.  Why? 
- Small far away galaxies (high redshifts) can't be detected with near infrared photometry because they are too faint
- So, at high redshift, the catalog is missing low‑mass galaxies simply because the telescope can’t see them, not because they don’t exist.

By inspection of this plot, we will only keep sources with log(M)>9 to remove sources which do not have complete coverage across redshift.

In [ ]:
print('number of datapoints originally = {}'.format(tab.shape[0]))
tab = tab[tab.lmass > 9].copy()
print('number of datapoints after drop = {}'.format(tab.shape[0]))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

ax.scatter(tab.z_spec, tab.lmass, alpha=0.2, color='grey')
ax.set_xlim(0, 2)
ax.set_ylim(7, 12)
ax.set_xlabel(r'$z_{\rm spec}$')
ax.set_ylabel(r'$\log{(M_{*})}\,\,[M_{\odot}]$');

## Define our target variable

We're interested in predicting redshift, so our "target" variable will be z_spec. The "features" will be other columns but specifically not the spectroscopic and photometric redshifts.

In [ ]:
target = 'z_spec'

Next, we will remove rows in our dataset where a spectroscopic redshift is not provided:

In [ ]:
print('number of datapoints originally = {}'.format(tab.shape[0]))
tab = tab[(tab[target] > 0)] 
print('number of datapoints after drop = {}'.format(tab.shape[0]))

## Transform categorical data into a numerical representation
Categorical variables need to be modified to become numerical values. We can do this using pandas and sklearn have different tools to accomplish this.  We will use `pd.get_dummies` to perform one hot encoding - a technique used to convert categorical featurs into a binary matrix of 1s and 0s.  Below is a simple illustration of what this will do:

![](https://miro.medium.com/v2/resize:fit:1400/1*ggtP4a5YaRx6l09KQaYOnw.png)

Now let's apply this function to our data and inspect `tab`.

In [ ]:
tab['field'].value_counts()

In [ ]:
tab = pd.get_dummies(tab, columns=['field'],dtype=float)
tab

## Defining Features
Next, we will define the features. 

Let's drop a few of the features. The 'Av', 'lmass' and 'z_peak' values were all computed via FAST photometric fit, and so we will exclude them as well. In addition, we will exclude the categorical flag variables ('flags', 'f140w_flag', 'star_flag', 'use_phot', 'near_star' and 'field').

In [ ]:
features = [col for col in tab.columns if (col != target)]
features = [col for col in features if (col != 'Av') and (col != 'lmass') and (col != 'z_peak') 
            and (col != 'flags') and (col != 'f140w_flag') and (col != 'star_flag') 
            and (col != 'use_phot') and (col != 'near_star') and (col != 'field') 
            and (col != 'field_UDS') and (col != 'ra') and (col != 'dec')]

In [ ]:
features

## Imputing missing values 
Finally we will impute missing values of photometric errors (set to -99. in the table) by assigning them the median of the distribution:

In [ ]:
# get columns associated with errors
errors = [col for col in features if (col[:1] == 'e') and (col[-1:] == 'W')]
errors

In [ ]:
# impute values with median
for error in errors:
    tab[error] = np.where(tab[error] < -90, tab[error].median(), tab[error])

Finally lets inspect distribution of features

In [ ]:
fig = plt.figure(0, [20, 18])

for k, feat in enumerate(features):
    ax = fig.add_subplot(7, 6, k+1)
    ax.hist(tab[feat], bins=50, log=True, color='navy')
    ax.set_title(feat)

ax = fig.add_subplot(7, 6, len(features)+1)
ax.hist(tab[target], bins=50, color='red')
ax.set_title(r'$z_{\rm spec}$')

plt.tight_layout()
plt.show()

## Build and Evaluate Baseline Models 

When building machine learning models its good to start with a baseline model.  This will give you a place to reference back to in terms of the evaluation metrics you are computing. In this notebook, we will start with a few simple things just to get a feel of what performance we can expect from our models:

1. **Mean Regressor:** we predict our target is the mean of all the target values contained in our dataset
2. Multiple Linear regression with no feature engineering


### Test / Train Split 

Let's get started by splitting the data we have into three groups:

1. Training: Data used to train the model
2. Testing: Data used to evaluate models performance when performing model tuning
3. Validation: Data used to evaluate the models performance on unseen data points 

In [ ]:
# Define X (features ) and y (target) for the dataset we are using
X = tab[features]
y = tab[target]

# first reserve 70% of the data for training, 30% for validation
X_train, X_validate, y_train, y_validate= train_test_split(X, y,  
                                                           test_size=0.3, 
                                                           random_state=42)

# second, split the validation set in half to obtain validation and test sets. 
X_validate, X_test, y_validate, y_test = train_test_split(X_validate, 
                                                          y_validate,  
                                                          test_size=0.5, 
                                                          random_state=42)

### Mean Regressor 

Next, lets build our mean regressor.  This can be done using sklearn's `DummyRegressor` class. 

In [ ]:
from sklearn.dummy import DummyRegressor
dummy_regr = DummyRegressor(strategy="mean")
dummy_regr.fit(X_train, y_train)

dummy_regr.predict(X_test)[0:10]

In [ ]:
# evaluate model 

# get predictions
y_predict = dummy_regr.predict(X_test) 
# compute mse with sklearns built in function
mse_dummy_regr= mean_squared_error(y_test, y_predict )
print('Mean squared error for mean regressor baseline: {}'.format(mse_dummy_regr))

## Naive Multiple Linear Regression with no feature engineering / inspection

Another thing we can do is just try multiple linear regression with our data.  I wouldn't expect good results here, but by comparing the results of this with the mean regressor will give us a sense for if there is signal in our data as the features currently are for linear regression.  

When building linear regression models, we have a choice of whether we standardize our data.  There are pros to either decision:

1. **Standardize data:** All of our features are on the same scale which means we can compare our learned coefficients.  This could give us insight into which of our features are the most helpful for making predictions.  However, we can interpret slopes 
2. **Do not standardize data:** We maintain the interpretation of coefficients

I would like to use this initial pass to gain some instinct for what features are most important, so I will standardize our data and then compare the coefficients using `StandardScaler`.  `StandardScaler` transforms columns of your data to have mean 0 and standard deviation 1. 

In [ ]:
from sklearn.linear_model import LinearRegression

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

linreg = LinearRegression().fit(X_train_scaled, y_train)

# evaluate model 
y_predict = linreg.predict(X_test_scaled) # get predictions
mse_mlr=mean_squared_error(y_test, y_predict )
print('Mean squared error for multiple linear regression baseline: {}'.format(mse_mlr))

Since we standardized our data we can compare the coefficients to see what feature is playing the biggest role in making predictions. Below I place the coefficients in to a dataframe.

In [ ]:
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': linreg.coef_, 'importance':np.absolute(linreg.coef_)})
coef_df.sort_values(by='importance',ascending=False)

Now, lets take the information above and see if we can use it to dive deeper into a few features

## Simple Linear Regression with Feature Engineering

Let's explore some of our top performing feature based on analysis above: f_F125W. First let's just see what the relationship looks like between feature and target.  

**Exercise**: Try uncommenting log scales and see what you discover.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(tab.f_F125W,tab.z_spec,alpha=0.1)
ax.set_xlabel('f_F125W')
ax.set_ylabel('z spec')

################################
# Uncomment lines to see how plot changes 
################################
#ax.set_yscale('log')
#ax.set_xscale('log')

Based on data visualization above, lets engineer some new features and see what kind of performance that will give use.  In this example we will, take the log of both the feature and target.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,5))
ax[0].scatter(tab.f_F125W,tab.z_spec,alpha=0.1)
ax[1].scatter(np.log(tab.f_F125W),np.log(tab.z_spec),alpha=0.1)
ax[0].set_xlabel('f_F125W')
ax[0].set_ylabel('z spec')
ax[1].set_xlabel('log(f_F125W)')
ax[1].set_ylabel('log(z spec)')
#ax = scatter_matrix(tab[['f_F140W','f_F606W', 'z_spec']], ax=ax) #, diagonal='kde')

Before we engineer this new feature, we will want to make sure there are no negative values in the 'f_F125W' column.

In [ ]:
# when taking logs its important to ensure no negative values.  Lets see how many negative values we have
(tab[['f_F125W']] < 0).value_counts()

Since its a small fraction of the data, lets drop these values

In [ ]:
tab = tab[tab['f_F125W'] > 0]

Now lets build a a few models with simple linear regression where we engineer our feature and target; in this example we take the log of both the feature and target


In [ ]:
# Do Another test train split with only the f_F125W column
X_train, X_test, y_train, y_test = train_test_split(tab[['f_F125W']], 
                                                    tab.z_spec,
                                                    random_state = 0)

simplelinreg_log = LinearRegression().fit(np.log(X_train), np.log(y_train))

Next lets evaluate our model.  It is important to note that if we are predicting the log(z_spec) with our linear regression model, we will need to convert the models prediction back to z_spec by using `np.exp()`. 

In [ ]:
# evaluate linreg_log model 
y_predict = np.exp(simplelinreg_log.predict(np.log(X_test))) # get note, we undo the log of the target by using np.exp
mse_linreg_log = mean_squared_error(y_test, y_predict )

# compare performance of our simple linear regression models with our baselines
print('Mean squared error for mean regressor baseline: {}'.format(mse_dummy_regr))
print('Mean squared error for multiple linear regression baseline: {}'.format(mse_mlr))
print('The MSE for linreg is {}'.format(mse_linreg_log))

OK, nice! By using simple linear regression with engineering features, we now have a better score than our initial pass with MLR.  

Next, we will visualize our models prediction against the data below.

In [ ]:
fig,ax = plt.subplots(2)

logx = np.log(tab[['f_F125W']])
logy= np.log(tab.z_spec)
ax[0].scatter(logx, logy, marker= 'o', s=50, alpha=0.2, label='training data')
ax[0].plot(logx, simplelinreg_log.coef_ * logx+ simplelinreg_log.intercept_, 'r-', label='model')
#ax[0].set_title('Least-squares linear regression')
ax[0].set_xlabel('Log( Feature value (x) )')
ax[0].set_ylabel('Log( Target value (y))')
ax[0].legend()

# values for drawing line
x = np.linspace(-2,10,100)
y_predictions = np.exp( simplelinreg_log.coef_ * x+ simplelinreg_log.intercept_ )

ax[1].scatter(tab.f_F125W, tab.z_spec, marker= 'o', s=50, alpha=0.2, label='training data')
ax[1].plot(np.exp(x), y_predictions, 'r-', label='model')
#ax[1].set_title('Least-squares linear regression')
ax[1].set_xlabel('Feature value (x)')
ax[1].set_ylabel('Target value (y)')
ax[1].legend()

fig.tight_layout()

## Incorporate engineered features into our MLR model

First we will copy our dataframe for future feature engineering and pull out features based on the initial linear regression model we recreated earlier.

In [ ]:
# engineer new features from simple linear regression example 
tab_engineer = tab.copy()
tab_engineer[target] = np.log(tab_engineer[target])
tab_engineer['f_F125W'] = np.log(tab_engineer['f_F125W'])

# plot new scatter matrix

pd.plotting.scatter_matrix(tab_engineer[['f_F125W','f_F160W','tot_cor','kron_radius']+[target]],figsize=(7,7))
plt.tight_layout();

Now lets use our new feature list and see if we get any performance improvement for our linear regression model with additional features.

In [ ]:
X = tab_engineer[features]
y = tab_engineer[target]

# first reserve 70% of the data for training, 30% for validation
X_train, X_validate, y_train, y_validate= train_test_split(X, y,test_size=0.3, random_state=42)

# second, split the validation set in half to obtain validation and test sets. 
X_validate, X_test, y_validate, y_test = train_test_split(X_validate, y_validate, test_size=0.5, random_state=42)

# Scale data so we can compare slopes 
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

linreg_eng = LinearRegression().fit(X_train_scaled, y_train)

# evaluate model 
y_predict = np.exp(linreg_eng.predict(X_test_scaled))# get predictions
mse_mlr_eng=mean_squared_error(np.exp(y_test), y_predict )
print('Mean squared error for multiple linear regression baseline plus 1 engineerged feature and target: {}'.format(mse_mlr_eng))

In [ ]:
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': linreg_eng.coef_, 'importance':np.absolute(linreg_eng.coef_)})
coef_df.sort_values(by='importance',ascending=False)

# Summary of what we have done so far...

In [ ]:
print('Mean squared error for mean regressor baseline: {}'.format(mse_dummy_regr))
print('Mean squared error for multiple linear regression baseline: {}'.format(mse_mlr))
print('Mean squared error for simple linear regression is {}'.format(mse_linreg_log))
print('Mean squared error for multiple linear regression baseline plus 1 engineerged feature and target: {}'.format(mse_mlr_eng))

In [ ]:
# compare error to z_peak and see how far we have to go 
print("Best score from Astronomy Paper {}".format(mean_squared_error(tab.loc[X_test.index].z_spec,tab.loc[X_test.index].z_peak)))

## Exercise:

We have improved performance throughout this notebook. Using your astronomy expertise try to modify features, data processing pipeline, etc. and see if you can continu to improve performance of our linear regression model. 

## OPTIONAL: Explore Regularization

Finally, let's see if we use some regularization techniques if that improves performance of our model.  When using a regularization technique it is important to explore the hyper paramter $\alpha$.  In the code below we will build several models for both Ridge and Lasso regresion using different values of $\alpha$.

In [ ]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn import preprocessing

alphas = np.logspace(-7, 7, 11)
models_lasso={}
models_ridge={}
for alpha in alphas:
    # build model
    model = Lasso(alpha=alpha, max_iter=100000)
    model.fit(X_train_scaled, y_train)
    models_lasso[alpha]=model

    # build model
    modelr = Ridge(alpha=alpha, max_iter=1000000)
    modelr.fit(X_train_scaled, y_train)
    models_ridge[alpha]=modelr

Next, we will evaluate these models with our testing data and plot how our performance changes with $\alpha$.

In [ ]:
# function to evaluate models 
def evaluate(model,X_test_scaled,y_test):
    y_predict = np.exp(model.predict(X_test_scaled)) # get predictions and undo log of target
    return mean_squared_error(np.exp(y_test), y_predict ) # compute MSE

# compute MSE for all values of alpha
mse_array_lasso = [evaluate(model,X_test_scaled,y_test) for model in models_lasso.values()]
mse_array_ridge = [evaluate(model,X_test_scaled,y_test) for model in models_ridge.values()]

In [ ]:
# plot results and print performance of best model
fig,ax=plt.subplots()
ax.plot(alphas,mse_array_lasso,label='lasso')
ax.scatter(alphas,mse_array_lasso)

ax.plot(alphas,mse_array_ridge,label='ridge')
ax.scatter(alphas,mse_array_ridge)

ax.scatter(alphas[np.argmin(mse_array_lasso)],np.array(mse_array_lasso).min(), c='k', s=200,alpha=0.5,label='Best score lasso')
ax.scatter(alphas[np.argmin(mse_array_ridge)],np.array(mse_array_ridge).min(), c='r', s=100,alpha=0.5,label='Best score ridge')

ax.legend()
ax.set_xscale('log')
ax.set_ylabel('Mean Squared Error')
ax.set_xlabel("$\\alpha$")

print("Best score from Ridge Regression {}".format(mse_array_ridge[np.argmin(mse_array_ridge)]))
print("Best score from Lasso Regression {}".format(mse_array_lasso[np.argmin(mse_array_lasso)]))

In this example, we found no performance gains from using ridge and lasso regression. 